<a href="https://colab.research.google.com/github/JuanJRojas/IA-y-Minirobots-JJR-y-AFM/blob/main/Cuarta%20Actividad/Tarea4IA%26Mini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Punto 1

Descargue MEPX, https://www.mepx.org/, estúdielo y corra uno de los ejemplos que trae:

![Punto1IA4](https://drive.google.com/uc?id=1sbkRbOHCXUtb3NhDaSTSOW4eUZEqcTmz)


##Punto 3

Este ejercicio consiste en obtener a partir de programación genética una serie de operaciones, que dadas 10 posibles entradas de 4 números binarios, resulten en 10 posibles salidas de 7 números binarios como salida que se manifesten fenotípicamente como un número decimal en el display siete segmentos. La solución, las operaciones lógicas necesarias se pueden obtener por métodos como los mapas de Karnaugh.

Las terminales del sistema corresponden a las 4 posibles entradas, de estas dependen todos los posibles resultados, x0, x1, x2 y x3. Mientras las funciones que modifican el valor de estas entradas son todas operaciones lógicas como el not, el and, el or y el xor. la tabla de verdad que compara tanto la entrada como la salida se muestra a continuación.

| Entradas |    |    |     |
|----------|----|----|-----|
| x0       | x1 | x2 | x3  |
| 0        | 0  | 0  | 0   |
| 1        | 0  | 0  | 0   |
| 0        | 1  | 0  | 0   |
| 1        | 1  | 0  | 0   |
| 0        | 0  | 1  | 0   |
| 1        | 0  | 1  | 0   |
| 0        | 1  | 1  | 0   |
| 1        | 1  | 1  | 0   |
| 0        | 0  | 0  | 1   |
| 1        | 0  | 0  | 1   |


A continuación, las salidas:

| Salidas |   |   |   |   |   |    |   |
|---------|---|---|---|---|---|----|---|
| a       | b | c | d | e | f | g  |   |
| 1       | 1 | 1 | 1 | 1 | 1 | 0  |   |
| 0       | 1 | 1 | 0 | 0 | 0 | 0  |   |
| 1       | 1 | 0 | 1 | 1 | 0 | 1  |   |
| 1       | 1 | 1 | 1 | 0 | 0 | 1  |   |
| 0       | 1 | 1 | 0 | 0 | 1 | 1  |   |
| 1       | 0 | 1 | 1 | 0 | 1 | 1  |   |
| 1       | 0 | 1 | 1 | 1 | 1 | 1  |   |
| 1       | 1 | 1 | 0 | 0 | 0 | 0  |   |
| 1       | 1 | 1 | 1 | 1 | 1 | 1  |   |
| 1       | 1 | 1 | 0 | 0 | 1 | 1  |   |


La función de aptitud entonces toma los distintos casos que se pueden presentar para cada entrada y los compara con su respectiva salida. Le proporcionaría un puntaje más positivo a aquellos valores que se encuentren más cercanos al mapa de salida, haciedno una comparación uno a uno de los bits o del representante decimal del número binario. A continuación el intento del algoritmo con la librería DEAP:


In [3]:
import random
!pip install deap
import numpy as np
from deap import base, creator, tools, algorithms

# Mapeo de entradas (0 a 9) a sus correspondientes salidas de 7 segmentos
seven_segment_map = {
    (0, 0, 0, 0): (1, 1, 1, 1, 1, 1, 0),  # 0
    (0, 0, 0, 1): (0, 1, 1, 0, 0, 0, 0),  # 1
    (0, 0, 1, 0): (1, 1, 0, 1, 1, 0, 1),  # 2
    (0, 0, 1, 1): (1, 1, 1, 1, 0, 0, 1),  # 3
    (0, 1, 0, 0): (0, 1, 1, 0, 0, 1, 1),  # 4
    (0, 1, 0, 1): (1, 0, 1, 1, 0, 1, 1),  # 5
    (0, 1, 1, 0): (1, 0, 1, 1, 1, 1, 1),  # 6
    (0, 1, 1, 1): (1, 1, 1, 0, 0, 0, 0),  # 7
    (1, 0, 0, 0): (1, 1, 1, 1, 1, 1, 1),  # 8
    (1, 0, 0, 1): (1, 1, 1, 1, 0, 1, 1),  # 9
}

# Crear el tipo de fitness y el individuo (en este caso, queremos minimizar el error)
creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.FitnessMin)

toolbox = base.Toolbox()

# Cada individuo ahora tiene 10 salidas de 7 bits (una por cada número de 0 a 9)
toolbox.register("attr_bool", random.randint, 0, 1)
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_bool, 70)  # 10 números * 7 segmentos
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

# Función de evaluación
def eval_individual(individual):
    total_error = 0
    for i, (input_bits, expected_output) in enumerate(seven_segment_map.items()):
        # Extraer la salida predicha para la entrada actual (7 bits)
        predicted_output = individual[i*7:(i+1)*7]
        # Comparar con la salida esperada
        error = sum(abs(predicted_output[j] - expected_output[j]) for j in range(7))
        total_error += error
    return total_error,

toolbox.register("evaluate", eval_individual)

# Operadores genéticos: mutación, cruce y selección
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutFlipBit, indpb=0.05)
toolbox.register("select", tools.selTournament, tournsize=3)

# Función para predecir la salida de 7 segmentos para una entrada específica
def predict(best_individual, input_bits):
    index = list(seven_segment_map.keys()).index(input_bits)
    return best_individual[index*7:(index+1)*7]

# Algoritmo genético
def main():
    random.seed(42)

    # Crear la población inicial
    population = toolbox.population(n=300)

    # Algoritmo genético
    result_population = algorithms.eaSimple(
        population,
        toolbox,
        cxpb=0.5,
        mutpb=0.2,
        ngen=40,
        verbose=True
    )

    # Encontrar el mejor individuo
    best_individual = tools.selBest(population, 1)[0]

    # Mostrar el resultado final
    print("Mejor individuo:", best_individual)
    print("Fitness:", eval_individual(best_individual))

    # Probar la salida para una entrada
    salida_predicha=[]
    for i,(input_bits,expected_output) in enumerate(seven_segment_map.items()):
        salida_predicha.append(predict(best_individual, input_bits))
        print(f"Para la entrada {input_bits}, la salida predicha es: {salida_predicha[i]}")

if __name__ == "__main__":
    main()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 kB 4.1 MB/s eta 0:00:00
gen	nevals
0  	300   
1  	182   
2  	197   
3  	188   
4  	165   
5  	177   
6  	154   
7  	180   
8  	180   
9  	175   
10 	190   
11 	194   
12 	168   
13 	173   
14 	186   
15 	162   
16 	175   
17 	188   
18 	180   
19 	187   
20 	185   
21 	191   
22 	181   
23 	191   
24 	188   
25 	188   
26 	180   
27 	177   
28 	183   
29 	167   
30 	184   
31 	163   
32 	178   
33 	188   
34 	190   
35 	170   
36 	174   
37 	204   
38 	185   
39 	194   
40 	184   
Mejor individuo: [1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1]
Fitness: (0,)
Para la entrada (0, 0, 0, 0), la salida predicha es: [1, 1, 1, 1, 1, 1, 0]
Para la entrada (0, 0, 0, 1), la salida predicha es: [0, 1, 1, 0, 0, 0, 0]
Para la entrada (0, 0, 1, 0), la salida predicha es: [1, 1, 0, 1, 

##Punto 4

El ejercicio 4 corresponde a un robot que "reparte galletas" a distintos miembros del equipo de ingenieros en una sala, cada que el robot reparte una galleta, este ganaría un punto, para así recompensarlo por pasar por los puntos donde se encuentran los objetivos.

los terminales son entonces las definiciones de las direcciones a las que se puede dirigir, ya sea abajo, derecha, arriba o izquierda, o sur, este, norte u oeste. Por su parte, las funciones se pueden definir por los "if - then" para cada una de las direcciones, y así determinar cuáles se acercan o no a los objetivos. Por último, la función de aptitud califica si las terminales elegidas en cada determinada función realizada se acerca al ingeniero y, de hacerlo, premiar al algoritmo con un valor positivo.

Si se desea que la IA sea más sofisticada, se debería modificar la función de aptitud para obtener más puntos por cada objetivo distinto por el que pasa, así como para obtener la mayor cantidad de puntos en la menor cantidad de movimientos posibles, según corresponda.

In [12]:
# Parámetros del entorno
SALA_SIZE = 10
NUM_INGENIEROS = 5
MOVIMIENTOS = ['N', 'S', 'E', 'O']  # Norte, Sur, Este, Oeste

# Generar posiciones aleatorias para los ingenieros
def generar_posiciones_ingenieros(num_ingenieros, sala_size):
    return [(random.randint(0, sala_size - 1), random.randint(0, sala_size - 1)) for _ in range(num_ingenieros)]

# Función de aptitud: contar cuántos ingenieros han recibido galleta
def funcion_aptitud(ruta, ingenieros, sala_size):
    x, y = sala_size // 2, sala_size // 2  # El robot comienza en el centro de la sala
    puntos = 0
    entregados = set()

    for movimiento in ruta:
        if movimiento == 'N':
            y = max(0, y - 1)
        elif movimiento == 'S':
            y = min(sala_size - 1, y + 1)
        elif movimiento == 'E':
            x = min(sala_size - 1, x + 1)
        elif movimiento == 'O':
            x = max(0, x - 1)

        if (x, y) in ingenieros and (x, y) not in entregados:
            puntos += 1
            entregados.add((x, y))

    return puntos

# Generar una población inicial de rutas aleatorias
def generar_poblacion(num_individuos, longitud_ruta):
    return [[random.choice(MOVIMIENTOS) for _ in range(longitud_ruta)] for _ in range(num_individuos)]

# Operadores genéticos: cruce y mutación
def cruce(parent1, parent2):
    punto_cruce = random.randint(1, len(parent1) - 1)
    return parent1[:punto_cruce] + parent2[punto_cruce:]

def mutacion(individuo):
    idx = random.randint(0, len(individuo) - 1)
    individuo[idx] = random.choice(MOVIMIENTOS)

# Selección por torneo
def seleccion_torneo(poblacion, puntuaciones, tamaño_torneo):
    seleccionados = random.sample(range(len(poblacion)), tamaño_torneo)
    mejor = max(seleccionados, key=lambda i: puntuaciones[i])
    return poblacion[mejor]

# Algoritmo Genético
def algoritmo_genetico(num_generaciones, tam_poblacion, longitud_ruta, tamaño_torneo, tasa_mutacion):
    ingenieros = generar_posiciones_ingenieros(NUM_INGENIEROS, SALA_SIZE)
    poblacion = generar_poblacion(tam_poblacion, longitud_ruta)

    for generacion in range(num_generaciones):
        puntuaciones = [funcion_aptitud(ind, ingenieros, SALA_SIZE) for ind in poblacion]
        mejor_individuo = poblacion[np.argmax(puntuaciones)]
        print(f"Generación {generacion + 1}: Mejor puntuación {max(puntuaciones)}, Ruta: {mejor_individuo}")

        nueva_poblacion = []
        for _ in range(tam_poblacion // 2):
            parent1 = seleccion_torneo(poblacion, puntuaciones, tamaño_torneo)
            parent2 = seleccion_torneo(poblacion, puntuaciones, tamaño_torneo)
            hijo1, hijo2 = cruce(parent1, parent2), cruce(parent2, parent1)
            if random.random() < tasa_mutacion:
                mutacion(hijo1)
            if random.random() < tasa_mutacion:
                mutacion(hijo2)
            nueva_poblacion.extend([hijo1, hijo2])

        # Elitismo: mantener el mejor individuo
        mejor_actual = poblacion[np.argmax(puntuaciones)]
        nueva_poblacion[random.randint(0, len(nueva_poblacion) - 1)] = mejor_actual

        poblacion = nueva_poblacion

    return mejor_individuo

# Parámetros del algoritmo
NUM_GENERACIONES = 150
TAM_POBLACION = 1000
LONGITUD_RUTA = 30
TAMAÑO_TORNEO = 5
TASA_MUTACION = 0.4

# Ejecutar el algoritmo genético
mejor_ruta = algoritmo_genetico(NUM_GENERACIONES, TAM_POBLACION, LONGITUD_RUTA, TAMAÑO_TORNEO, TASA_MUTACION)
print(f"Mejor ruta encontrada: {mejor_ruta}")

Generación 1: Mejor puntuación 4, Ruta: ['O', 'N', 'N', 'S', 'O', 'S', 'E', 'S', 'S', 'E', 'O', 'E', 'E', 'S', 'E', 'O', 'N', 'N', 'S', 'N', 'O', 'O', 'O', 'N', 'O', 'N', 'O', 'O', 'N', 'N']
Generación 2: Mejor puntuación 4, Ruta: ['O', 'N', 'N', 'S', 'O', 'S', 'E', 'S', 'S', 'E', 'O', 'E', 'E', 'S', 'E', 'O', 'N', 'N', 'S', 'N', 'O', 'O', 'O', 'N', 'O', 'N', 'O', 'O', 'N', 'N']
Generación 3: Mejor puntuación 4, Ruta: ['O', 'N', 'N', 'S', 'O', 'S', 'E', 'S', 'S', 'E', 'O', 'E', 'E', 'S', 'E', 'O', 'N', 'N', 'S', 'N', 'O', 'O', 'O', 'N', 'O', 'N', 'O', 'O', 'N', 'S']
Generación 4: Mejor puntuación 4, Ruta: ['S', 'E', 'N', 'N', 'N', 'O', 'E', 'S', 'S', 'S', 'S', 'O', 'S', 'O', 'S', 'N', 'N', 'N', 'O', 'N', 'N', 'E', 'N', 'O', 'S', 'E', 'O', 'O', 'O', 'O']
Generación 5: Mejor puntuación 4, Ruta: ['N', 'O', 'O', 'E', 'N', 'S', 'S', 'E', 'N', 'N', 'E', 'S', 'S', 'S', 'S', 'E', 'N', 'E', 'E', 'S', 'N', 'E', 'N', 'O', 'E', 'N', 'N', 'S', 'E', 'E']
Generación 6: Mejor puntuación 4, Ruta: ['E',